# 10 · DuckDB와 SQL 조회

Pandas로 작성한 결과와 SQL 결과를 비교하면서 조회, 필터, 집계, 조건 분류, 조인의 의미를 익힙니다. 이 노트북은 `duckdb` 패키지가 필요합니다.

> 위에서 아래로 실행하세요. 예제 데이터는 노트북 안에서 만듭니다. 코드 셀 아래의 출력으로 결과를 확인하고, 실제 데이터에서는 열 이름·단위·기간을 먼저 확인하세요.

## DataFrame 등록과 SELECT

**코드 → 코드 개념**: SQL의 `SELECT`는 열, `FROM`은 데이터 원천, `LIMIT`는 출력 행 수를 정한다.

**코드 사용법**: Pandas 표를 DuckDB에 등록해 조회한다.

In [ ]:
import pandas as pd
import duckdb
df = pd.DataFrame({"machine": ["A", "A", "B", "B"],
                   "cycle": [1, 2, 1, 2], "vibration": [2.1, 3.5, 2.4, 4.1]})
con = duckdb.connect()
con.register("measurements", df)
print(con.execute("SELECT machine, vibration FROM measurements LIMIT 3").df())
print(df[["machine", "vibration"]].head(3))

**같은 결과를 얻는 방법과 선택 이유**

- 이미 Python 표가 있으면 `register`로 SQL에서 사용한다. 파일에서 필요한 일부만 읽을 때는 `read_csv_auto('file.csv')`를 SQL의 `FROM`에 넣는다.
- `LIMIT 3`은 처음 세 행을 보여 주지만 순서가 보장되어야 하면 `ORDER BY`를 추가한다.

## WHERE와 ORDER BY

**코드 → 코드 개념**: `WHERE`는 집계 전 행을 거르고 `ORDER BY`는 결과를 정렬한다.

**코드 사용법**: 진동 3 이상을 내림차순으로 뽑는다.

In [ ]:
sql = con.execute("SELECT * FROM measurements WHERE vibration >= 3 ORDER BY vibration DESC").df()
pd_result = df.loc[df["vibration"] >= 3].sort_values("vibration", ascending=False).reset_index(drop=True)
print(sql, pd_result, sep="\n")

**같은 결과를 얻는 방법과 선택 이유**

- SQL은 파일·DB에서 필요한 행과 열만 먼저 줄이기 좋다. Pandas는 Python에서 후속 계산·그래프를 이어가기 좋다.
- SQL `NULL` 비교는 `= NULL`이 아니라 `IS NULL` 또는 `IS NOT NULL`을 쓴다.

## GROUP BY와 HAVING

**코드 → 코드 개념**: `GROUP BY`는 그룹 통계, `HAVING`은 집계 후 그룹 조건이다.

**코드 사용법**: 설비별 측정 건수와 평균을 비교한다.

In [ ]:
sql = con.execute("""
SELECT machine, COUNT(*) AS n, AVG(vibration) AS mean_vibration
FROM measurements
GROUP BY machine
HAVING COUNT(*) >= 2
ORDER BY machine
""").df()
pd_result = df.groupby("machine", as_index=False).agg(n=("vibration", "size"), mean_vibration=("vibration", "mean"))
print(sql, pd_result, sep="\n")

**같은 결과를 얻는 방법과 선택 이유**

- 행 조건은 `WHERE`, 그룹 통계 조건은 `HAVING`에 둔다. Pandas에서는 `groupby().agg()` 후 결과를 필터한다.
- `COUNT(*)`는 전체 행 수, `COUNT(vibration)`은 비결측 건수다. `AVG`는 결측을 제외한다.

## CASE WHEN과 Pandas 분류

**코드 → 코드 개념**: `CASE WHEN`은 SQL에서 조건에 따라 새 값을 만든다.

**코드 사용법**: 위험 라벨을 SQL과 Pandas에서 만든다.

In [ ]:
sql = con.execute("""
SELECT machine, cycle, vibration,
       CASE WHEN vibration >= 4 THEN 'danger'
            WHEN vibration >= 3 THEN 'watch'
            ELSE 'normal' END AS state
FROM measurements ORDER BY machine, cycle
""").df()
pd_result = df.assign(state=df["vibration"].map(lambda x: "danger" if x >= 4 else "watch" if x >= 3 else "normal"))
print(sql, pd_result, sep="\n")

**같은 결과를 얻는 방법과 선택 이유**

- 분류 규칙이 조회 결과의 일부면 SQL `CASE`가 자연스럽다. Python에서 규칙을 함수로 재사용할 때는 Pandas `map`·`np.select`를 쓴다.
- 조건 순서가 중요하다. `>=3`을 먼저 쓰면 4 이상의 값도 `watch`가 된다.

## JOIN과 키 검증

**코드 → 코드 개념**: 조인은 공통 키로 두 표를 연결한다. 키 중복은 예상보다 많은 행을 만든다.

**코드 사용법**: 설비 메타데이터를 붙이고 행 수를 확인한다.

In [ ]:
master = pd.DataFrame({"machine": ["A", "B"], "line": ["L1", "L2"]})
con.register("master", master)
joined = con.execute("""
SELECT m.machine, m.cycle, m.vibration, s.line
FROM measurements AS m LEFT JOIN master AS s USING (machine)
ORDER BY m.machine, m.cycle
""").df()
print(joined)
assert len(joined) == len(df)
assert not master["machine"].duplicated().any()

**같은 결과를 얻는 방법과 선택 이유**

- `LEFT JOIN`은 왼쪽 센서 행을 보존하고, `INNER JOIN`은 양쪽에 키가 있는 행만 남긴다.
- Pandas `merge(..., how='left', validate='many_to_one')`는 관계를 명시적으로 검증할 때 편하다. SQL 결과도 조인 전후 행 수와 키 유일성을 확인한다.

## 원본 학습 자료

[`2. practice/08_DuckDB`](../2.%20practice/08_DuckDB)